# Parameter Study Analysis

This notebook imports data collected by a Daphne parameter study, analyzes
scalability and performance metrics, and visualizes results.

## 1. Load Libraries and Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dask.dataframe as dd
%matplotlib inline

# Enable copy-on-write to follow pandas recommendation.
# see https://pandas.pydata.org/docs/user_guide/copy_on_write.html
pd.options.mode.copy_on_write = True

In [ ]:
# Load collected data from Parquet file.
# Replace '../data.parquet' with the path to your data file.
# As a reminder, please run `go run ./daphne study load` to produce this file
# Data is loaded in chunks and processed to categorize columns to limit peak
# memory usage.
delayedData = dd.read_parquet(
    '../../data.parquet',
    columns=['timestamp', 'mark', 'type', 'rid', 'bytesize', 'NumNodes', 'TxPerSecond']
)

def process_chunk(chunk):
    chunk = dd.from_delayed(chunk).compute()
    for col in ['mark', 'type']:
        chunk[col] = chunk[col].astype('category')
    return chunk

data = pd.concat((process_chunk(chunk) for chunk in delayedData.to_delayed()), ignore_index=True)

## 2. Number of Messages
The following chart illustrates the number of messages being transferred over
the network depending on the number of involved nodes and transaction rates.


In [ ]:
# Filter message sent and study started rows
msgsent = data[data['mark'] == 'MsgSent']
scenarios = data[data['mark'] == 'StudyStarted'][['rid', 'NumNodes', 'TxPerSecond']]

# Count messages per experiment run
msg_counts = msgsent.groupby('rid').size().reset_index(name='MsgCount')

# Join with experiment parameters
merged = pd.merge(msg_counts, scenarios, on='rid')

# Group by NumNodes and TxPerSecond, calculate mean and std
stats = merged.groupby(['NumNodes', 'TxPerSecond'])['MsgCount'].agg(['mean', 'std', 'count']).reset_index()

# Pivot for grouped bar plot
pivot = stats.pivot(index='NumNodes', columns='TxPerSecond', values='mean')
pivot_std = stats.pivot(index='NumNodes', columns='TxPerSecond', values='std')

# Get number of runs per group
pivot_counts = stats.pivot(index='NumNodes', columns='TxPerSecond', values='count')
min_counts = pivot_counts.min().min()
max_counts = pivot_counts.max().max()
num_runs_text = f'runs per group: min={min_counts}, max={max_counts}'

# Plot grouped bar chart with error bars
pivot.plot(kind='bar', yerr=pivot_std, figsize=(10, 6), capsize=4)
plt.title('Messages Transferred by NumNodes and Transaction Rate (' + num_runs_text + ')')
plt.xlabel('NumNodes')
plt.ylabel('Mean Number of Messages Transferred')
plt.legend(title='Transaction Rate (TxPerSecond)')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
max_tx_per_second = data['TxPerSecond'].max()

msgsent = data[data['mark'] == 'MsgSent']
grouped = msgsent.groupby(['rid', 'type'], observed=True).size().reset_index(name='Count')

scenarios = data[data['mark'] == 'StudyStarted'][['rid', 'NumNodes', 'TxPerSecond']]
scenarios = scenarios[scenarios['TxPerSecond'] == max_tx_per_second]
grouped = pd.merge(grouped, scenarios, on='rid')

pivoted = grouped.pivot_table(
    index=['NumNodes', 'TxPerSecond'], 
    columns='type', 
    values='Count', 
    fill_value=0, 
    observed=True,
    aggfunc='mean',
)
pivoted = pivoted.reset_index()

# Plot stacked bars for each TxPerSecond value
pivoted.set_index('NumNodes').drop(columns=['TxPerSecond']).plot(kind='bar', stacked=True, figsize=(12, 7), alpha=0.7)

plt.title('Message Share breakdown by NumNodes for Tx-Rate of ' + str(max_tx_per_second) + ' TPS')
plt.xlabel('NumNodes')
plt.ylabel('Total Messages Sent')
plt.legend(title='Message Type')
plt.grid(True)
plt.show()

## 3. Network Bandwidth Usage
The following chart displays the total data transferred in the network (in bytes) depending on the number of nodes and the transaction rate.

In [ ]:
# Filter message sent and study started rows
msgsent = data[data['mark'] == 'MsgSent']
scenarios = data[data['mark'] == 'StudyStarted'][['rid', 'NumNodes', 'TxPerSecond']]

# Sum bytesize per experiment run
msg_bytes = msgsent.groupby('rid')['bytesize'].sum().reset_index(name='TotalBytes')

# Join with experiment parameters
merged_bytes = pd.merge(msg_bytes, scenarios, on='rid')

# Group by NumNodes and TxPerSecond, calculate mean and std
stats_bytes = merged_bytes.groupby(['NumNodes', 'TxPerSecond'])['TotalBytes'].agg(['mean', 'std', 'count']).reset_index()

# Pivot for grouped bar plot
pivot_bytes = stats_bytes.pivot(index='NumNodes', columns='TxPerSecond', values='mean')
pivot_bytes_std = stats_bytes.pivot(index='NumNodes', columns='TxPerSecond', values='std')

# Get number of runs per group
pivot_bytes_counts = stats_bytes.pivot(index='NumNodes', columns='TxPerSecond', values='count')
min_bytes_counts = pivot_bytes_counts.min().min()
max_bytes_counts = pivot_bytes_counts.max().max()
num_bytes_text = f'runs per group: min={min_bytes_counts}, max={max_bytes_counts}'

# Plot grouped bar chart with error bars
pivot_bytes.plot(kind='bar', yerr=pivot_bytes_std, figsize=(10, 6), capsize=4)
plt.title('Total Bytes Transferred by NumNodes and Transaction Rate (' + num_bytes_text + ')')
plt.xlabel('NumNodes')
plt.ylabel('Mean Total Bytes Transferred')
plt.legend(title='Transaction Rate (TxPerSecond)')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Filter MsgSent rows and group by rid and type for memory usage
memsent = data[data['mark'] == 'MsgSent']
memsent = memsent[memsent['type'].notna()]

# Calculate total bytes sent per message type per experiment run
mem_grouped = memsent.groupby(['rid', 'type'], observed=True)['bytesize'].sum().reset_index(name='TotalBytes')

# Join with experiment parameters for max TxPerSecond
mem_scenarios = scenarios[scenarios['TxPerSecond'] == max_tx_per_second]
mem_grouped = pd.merge(mem_grouped, mem_scenarios, on='rid')

# Pivot for stacked bar plot
mem_pivoted = mem_grouped.pivot_table(
    index=['NumNodes', 'TxPerSecond'],
    columns='type',
    values='TotalBytes',
    fill_value=0,
    observed=True,
    aggfunc='mean'
).reset_index()

# Plot stacked bars for each TxPerSecond value
mem_pivoted.set_index('NumNodes').drop(columns=['TxPerSecond']).plot(
    kind='bar', stacked=True, figsize=(12, 7), alpha=0.7
)

plt.title(f'Memory Share breakdown by NumNodes for Tx-Rate of {int(max_tx_per_second)} TPS')
plt.xlabel('NumNodes')
plt.ylabel('Total Bytes Sent')
plt.legend(title='Message Type')
plt.grid(True)
plt.show()